# Hex Grid Converter

Resample an image onto a hexagonal lattice (pointy-top), so each hexagon is one 'pixel'. Hex grids are more isotropic than square grids, so diagonals and curves read more smoothly. Mirrors `shaders/hexagonal_pixel_grid.frag`.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Make the repo's tools importable from notebooks/.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'tools')))
import halftone_common as hc

def demo_image(size=256):
    """A synthetic test image: smooth ramp + gradient disk, for when you
    don't have a photo handy. Replace with hc.load_image('path.jpg')."""
    y, x = np.mgrid[0:size, 0:size] / size
    ramp = x
    disk = np.clip(1.2 - 2.0 * np.hypot(x - 0.5, y - 0.5), 0, 1)
    g = np.clip(0.5 * ramp + 0.5 * disk, 0, 1)
    return np.stack([g, np.roll(g, 20, 0), 1 - g], axis=-1)

img = demo_image()
plt.figure(figsize=(4,4)); plt.imshow(img); plt.title('source'); plt.axis('off'); plt.show()


## Assign every pixel to its nearest hex center

In [ ]:
def hexify(img, size, pointy=True):
    h, w = img.shape[:2]
    ys, xs = np.mgrid[0:h, 0:w].astype(float)
    if not pointy:
        xs, ys = ys, xs  # swap axes for flat-top
    gx, gy = size*np.sqrt(3), size*1.5
    ax = (np.floor(xs/gx)+0.5)*gx; ay = (np.floor(ys/gy)+0.5)*gy
    bx = (np.floor((xs-gx/2)/gx)+0.5)*gx+gx/2
    by = (np.floor((ys-gy/2)/gy)+0.5)*gy+gy/2
    da_ = (xs-ax)**2+(ys-ay)**2; db_ = (xs-bx)**2+(ys-by)**2
    pb = db_ < da_
    cx = np.where(pb, bx, ax); cy = np.where(pb, by, ay)
    if not pointy: cx, cy = cy, cx
    cyi = np.clip(cy.astype(int), 0, h-1); cxi = np.clip(cx.astype(int), 0, w-1)
    return img[cyi, cxi]

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, s in zip(ax, [4, 8, 14, 24]):
    a.imshow(hexify(img, s)); a.set_title(f'hex size {s}'); a.axis('off')
plt.show()

## Square vs hex at matched cell area
Same nominal resolution, different lattice — note how curves soften on the hex grid.

In [ ]:
def squareify(img, size):
    h, w = img.shape[:2]
    ys, xs = np.mgrid[0:h, 0:w]
    cy = np.clip((ys//size)*size + size//2, 0, h-1)
    cx = np.clip((xs//size)*size + size//2, 0, w-1)
    return img[cy, cx]

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(squareify(img, 12)); ax[0].set_title('square'); ax[0].axis('off')
ax[1].imshow(hexify(img, 8)); ax[1].set_title('hex'); ax[1].axis('off')
plt.show()

## Pointy-top vs flat-top orientation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(hexify(img, 12, pointy=True));  ax[0].set_title('pointy-top'); ax[0].axis('off')
ax[1].imshow(hexify(img, 12, pointy=False)); ax[1].set_title('flat-top'); ax[1].axis('off')
plt.show()